# 5. Validação: os detectores de risco são confiáveis?

**Pergunta:** quando o código diz "esta skill tem `curl | sh`" ou "esta skill lida com segredos", ele está certo?

**Por que este notebook existe:** todos os números do notebook 04 (6,4% de gravidade alta, 17,3% de variantes que acrescentam capacidade, McNemar) vêm de **expressões regulares**, que procuram palavras no texto. Uma regex não entende contexto:

| Trecho | A regex marca | Na verdade é |
|---|---|---|
| "Instale com `curl -sSL https://… \| bash`" | pipe_to_shell | **instrução real** |
| "Nunca coloque a API key no código" | secrets | só uma **menção** (é até um conselho de segurança) |
| "o arquivo `.env.example` lista as variáveis" | secrets | menção, sem acesso a segredo |

Se a maioria das detecções de uma categoria for menção, a prevalência daquela categoria está **superestimada**. Precisamos medir a **precisão** de cada detector: de tudo que ele marcou, quanto é instrução de verdade.

**Por que uma amostra:** não dá para ler 1,6 milhão de skills à mão. Sorteamos aleatoriamente alguns trechos detectados em cada categoria, rotulamos esses à mão e usamos a proporção correta na amostra como **estimativa** da precisão do detector, com intervalo de confiança. É o procedimento padrão para validar heurísticas em estudos de mineração de repositórios.

**Etapas:** 5.1 sortear a amostra → 5.2 rotular à mão (trabalho humano, fora do notebook) → 5.3 calcular a precisão → 5.4 medir a concordância entre dois avaliadores → 5.5 ver o efeito nos números do notebook 04.

**Este notebook não está quebrado: ele está à espera da rotulagem.** Enquanto a coluna `label` do CSV estiver vazia, as seções 5.3–5.5 mostram só o status. Elas se preenchem sozinhas quando o CSV for rotulado e o notebook rodar de novo.

---

### English summary
Regular expressions detect a *mention*, not an *instruction*. We draw a stratified random sample (seed 42) of matched snippets per capability and label each by hand as `instructs`, `mentions` or `false_match`. Precision = `instructs` / labelled, reported with 95% Wilson intervals; inter-rater agreement with Cohen's kappa.

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("GITSKILLS_DB", str(ROOT / "data/raw/agent_skills_release.db"))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 80)
from gitskills import clean, db, reuse, adoption, patterns
from gitskills.plots import save, save_table
%matplotlib inline
print("database:", os.environ["GITSKILLS_DB"])
import re
from statsmodels.stats.proportion import proportion_confint

## 5.1 Sortear a amostra
**Objetivo:** para cada uma das 9 categorias, sortear 12 skills em que a regex disparou (108 trechos no total) e guardar ~160 caracteres antes e depois do ponto detectado, para dar contexto a quem rotula. A semente fixa (42) faz o sorteio ser reproduzível. O arquivo só é criado se ainda não existir, para nunca apagar rótulos já feitos.

In [ ]:
PER_CATEGORY = 12   # 9 categories x 12 = 108 snippets
skills = clean.strict(clean.load_skills())
rows = []
for cap, rx in patterns.COMPILED_RISKS.items():
    hits = skills[skills[f"risk_{cap}"].fillna(False)]
    pick = hits.sample(min(PER_CATEGORY, len(hits)), random_state=42)
    texts = db.query(f"SELECT file_sha, content FROM artifacts WHERE dedup_primary = 1 AND file_sha IN ({','.join('?' * len(pick))})",
                     tuple(pick["file_sha"]))
    for _, t in texts.iterrows():
        m = rx.search(t["content"] or "")
        if m:
            s = max(0, m.start() - 160); e = min(len(t["content"]), m.end() + 160)
            rows.append({"capability": cap, "file_sha": t["file_sha"], "snippet": t["content"][s:e].replace("\n", " "), "label": "", "label_2": ""})
sample = pd.DataFrame(rows)
out = ROOT / "results/tables/validation_sample.csv"
if not out.exists():
    sample.to_csv(out, index=False)
print(f"{len(sample)} snippets -> {out}")
sample.groupby("capability").size().rename("snippets").to_frame()

## 5.2 Como rotular (trabalho manual)
Abra `results/tables/validation_sample.csv` (Excel, Numbers ou Google Sheets), leia o `snippet` de cada linha e preencha a coluna **`label`** com uma de três palavras:

* **`instructs`**: o texto manda o agente executar a ação ("rode `curl … | bash`", "leia o arquivo `.env`");
* **`mentions`**: o texto fala da ação sem mandar executar ("nunca exponha a API key", "evite `rm -rf`");
* **`false_match`**: a regex casou com outra coisa ("token" no sentido de LLM, `--force` de outra ferramenta).

**Segundo avaliador (recomendado):** outra pessoa preenche a coluna **`label_2`** de forma independente, sem ver a `label`. Isso mede se o critério é claro (5.4). Depois as divergências são discutidas e a `label` final fica com o consenso.

Salve como CSV, sem mudar os nomes das colunas, e rode o notebook de novo.

## 5.3 Precisão por categoria
**Objetivo:** estimar, para cada categoria, a fração de detecções que são instruções reais, com intervalo de confiança de Wilson a 95%.

In [ ]:
VALID = {"instructs", "mentions", "false_match"}
lab = pd.read_csv(ROOT / "results/tables/validation_sample.csv", dtype=str, keep_default_na=False)
if "label_2" not in lab:
    lab["label_2"] = ""
lab["label"] = lab["label"].str.strip().str.lower()
lab["label_2"] = lab["label_2"].str.strip().str.lower()
bad = lab[(lab["label"] != "") & ~lab["label"].isin(VALID)]
done = lab[lab["label"].isin(VALID)]
print(f"labelled: {len(done)}/{len(lab)} snippets" + (f"; {len(bad)} with an unknown label: {sorted(bad['label'].unique())}" if len(bad) else ""))
if done.empty:
    print("STATUS: waiting for manual labels (see 5.2). Precision cannot be computed yet.")
    prec = None
else:
    g = done.groupby("capability")["label"]
    prec = pd.DataFrame({"labelled": g.size(), "instructs": g.apply(lambda s: int((s == "instructs").sum()))})
    prec["precision"] = prec["instructs"] / prec["labelled"]
    ci = [proportion_confint(k, n, method="wilson") for k, n in zip(prec["instructs"], prec["labelled"])]
    prec["ci_low"], prec["ci_high"] = [c[0] for c in ci], [c[1] for c in ci]
    save_table(prec.reset_index(), "validation_precision")
    display(prec.round(2))

**Como ler:** precisão 0,90 quer dizer que 9 em cada 10 detecções daquela categoria são instruções reais. Com 12 trechos por categoria, o intervalo de confiança é largo (cerca de ±25 pontos percentuais): serve para separar detectores bons de ruins, não para afinar a segunda casa decimal. Se uma categoria ficar abaixo de ~0,7, o número dela no notebook 04 deve ser reportado com ressalva, ou o detector deve ser refinado em `patterns.py`.

## 5.4 Concordância entre avaliadores (κ de Cohen)
**Objetivo:** verificar se duas pessoas, rotulando de forma independente, chegam ao mesmo rótulo. κ desconta a concordância que aconteceria por acaso: acima de 0,6 é considerado substancial, acima de 0,8 quase perfeito.

In [ ]:
both = lab[lab["label"].isin(VALID) & lab["label_2"].isin(VALID)]
if both.empty:
    print("STATUS: no snippet has both 'label' and 'label_2' yet; kappa needs a second rater.")
else:
    cats = sorted(VALID)
    po = (both["label"] == both["label_2"]).mean()
    pe = sum((both["label"] == c).mean() * (both["label_2"] == c).mean() for c in cats)
    kappa = (po - pe) / (1 - pe) if pe < 1 else float("nan")
    print(f"snippets rated twice: {len(both)}; raw agreement {po:.1%}; Cohen's kappa {kappa:.2f}")
    display(pd.crosstab(both["label"], both["label_2"]))

## 5.5 Efeito nos números do notebook 04
**Objetivo:** mostrar o que a precisão muda na prevalência. Prevalência ajustada ≈ prevalência detectada × precisão. Por exemplo, se 17,6% das skills disparam `secrets` e só metade das detecções é instrução real, a prevalência real de instruções que lidam com segredos é ~9%.

In [ ]:
strict = skills
detected = pd.Series({c: strict[f"risk_{c}"].mean() for c in patterns.COMPILED_RISKS}, name="detected_share")
if prec is None:
    print("STATUS: waiting for labels; showing detected prevalence only.")
    display(detected.to_frame().round(3))
else:
    adj = detected.to_frame().join(prec[["precision", "ci_low", "ci_high"]])
    adj["adjusted_share"] = adj["detected_share"] * adj["precision"]
    adj["adjusted_low"] = adj["detected_share"] * adj["ci_low"]
    adj["adjusted_high"] = adj["detected_share"] * adj["ci_high"]
    display(adj.round(3))

## Conclusão do notebook 05
Este é o passo de **validação** da metodologia: transforma "a regex disparou" em uma estimativa de quanto confiar em cada número da RQ5. Sem ele, os resultados do notebook 04 devem ser lidos como **limites superiores** (detecções, não instruções confirmadas). Com ele, cada prevalência pode ser reportada com a precisão do detector e, se houver dois avaliadores, com a concordância κ.